# 🔗 Módulo 7: Joins: unir tablas

merge, left/anti joins, self joins y concat. Pensar como en una base de datos.

**PandasQuest** · versión web interactiva: [https://rosalesluciano.github.io/pandas-quest/](https://rosalesluciano.github.io/pandas-quest/#/m/m07)

**Cómo funciona:**
1. Ejecuta la celda ⚙️ de abajo (carga las tablas y el corrector).
2. En cada ejercicio escribe tu código y guarda la respuesta en `resultado`.
3. Ejecuta la celda: `comprobar()` te dirá si está bien. 🎯

¿Atascado? `ver_solucion("id")` muestra la solución (¡pero inténtalo antes! 😉)

In [ ]:
#@title ⚙️ 1) Ejecuta esta celda primero (carga datos y corrector) { display-mode: "form" }
"""PandasQuest - motor de corrección.

Se usa igual en el navegador (Pyodide), en los notebooks de Colab y en los tests.
Compara el `resultado` del alumno contra el de la solución oficial con tolerancia
a diferencias irrelevantes (tipos int/float, nombres de índice, dtype string, etc.).
"""
import math

import numpy as np
import pandas as pd


def _norm_s(s):
    dt = s.dtype
    if isinstance(dt, pd.CategoricalDtype) or pd.api.types.is_string_dtype(dt):
        s = s.astype(object)
    elif pd.api.types.is_extension_array_dtype(dt):
        if pd.api.types.is_numeric_dtype(dt) and not pd.api.types.is_bool_dtype(dt):
            s = s.astype("float64")
        else:
            s = s.astype(object)
    if s.dtype == object:
        s = s.where(s.notna(), None)
    return s


def _norm_index(idx):
    if isinstance(idx, pd.MultiIndex):
        return idx
    if isinstance(idx, pd.CategoricalIndex) or pd.api.types.is_string_dtype(idx.dtype):
        return pd.Index(list(idx), dtype=object)
    return idx


def _norm(o):
    if isinstance(o, pd.Series):
        o = _norm_s(o.copy())
    else:
        cols = list(o.columns)
        o = pd.DataFrame({i: _norm_s(o.iloc[:, i]) for i in range(o.shape[1])}, index=o.index)
        o.columns = _norm_index(pd.Index(cols))
    o.index = _norm_index(o.index)
    return o


def _sort(o, ignore_index):
    if ignore_index:
        o = o.reset_index(drop=True)
        try:
            if isinstance(o, pd.Series):
                o = o.sort_values(kind="mergesort")
            else:
                o = o.sort_values(by=list(o.columns), kind="mergesort")
        except TypeError:
            key = o.astype(str) if isinstance(o, pd.Series) else o.astype(str).agg("|".join, axis=1)
            o = o.iloc[np.argsort(key.to_numpy(), kind="mergesort")]
        return o.reset_index(drop=True)
    try:
        return o.sort_index()
    except TypeError:
        return o


def _tipo(x):
    nombres = {pd.DataFrame: "un DataFrame", pd.Series: "una Series"}
    return nombres.get(type(x), f"un {type(x).__name__}")


def _scalar_eq(u, e):
    if isinstance(e, (np.generic,)):
        e = e.item()
    if isinstance(u, (np.generic,)):
        u = u.item()
    if isinstance(e, float) or isinstance(u, float):
        try:
            u, e = float(u), float(e)
        except (TypeError, ValueError):
            return False
        if math.isnan(e):
            return math.isnan(u)
        return math.isclose(u, e, rel_tol=1e-6, abs_tol=1e-6)
    if isinstance(e, (list, tuple)) and isinstance(u, (list, tuple, np.ndarray, pd.Index)):
        u = list(u)
        return len(u) == len(e) and all(_scalar_eq(a, b) for a, b in zip(u, e))
    try:
        return bool(u == e)
    except Exception:
        return False


def pq_compare(u, e, ordered=True, ignore_index=False):
    """Devuelve (ok, mensaje)."""
    if isinstance(e, (pd.DataFrame, pd.Series)):
        if not isinstance(u, type(e)):
            extra = ""
            if isinstance(e, pd.Series) and isinstance(u, pd.DataFrame):
                extra = " Pista: con una sola columna usa df['col'] (corchetes simples)."
            if isinstance(e, pd.DataFrame) and isinstance(u, pd.Series):
                extra = " Pista: usa doble corchete df[['col']] o .reset_index() / .to_frame()."
            return False, f"Tu resultado es {_tipo(u)}, pero se esperaba {_tipo(e)}.{extra}"
        if isinstance(e, pd.DataFrame):
            uc, ec = [str(c) for c in u.columns], [str(c) for c in e.columns]
            if uc != ec:
                if sorted(uc) == sorted(ec):
                    return False, f"Tienes las columnas correctas pero en otro orden. Se esperaba: {ec}"
                faltan = [c for c in ec if c not in uc]
                sobran = [c for c in uc if c not in ec]
                msg = "Las columnas no coinciden."
                if faltan:
                    msg += f" Faltan: {faltan}."
                if sobran:
                    msg += f" Sobran: {sobran}."
                return False, msg
        if u.shape != e.shape:
            return False, f"Forma distinta: tu resultado tiene {u.shape} y se esperaba {e.shape} (filas, columnas)."
        u2, e2 = _norm(u), _norm(e)
        if not ordered:
            u2, e2 = _sort(u2, ignore_index), _sort(e2, ignore_index)
        elif ignore_index:
            u2, e2 = u2.reset_index(drop=True), e2.reset_index(drop=True)
        kw = dict(check_dtype=False, check_names=False, check_index_type=False,
                  check_exact=False, rtol=1e-6, atol=1e-6)
        try:
            if isinstance(e2, pd.DataFrame):
                pd.testing.assert_frame_equal(u2, e2, check_column_type=False, check_freq=False, **kw)
            else:
                pd.testing.assert_series_equal(u2, e2, check_freq=False, **kw)
        except AssertionError as err:
            txt = str(err)
            if "index" in txt.lower() and "values are different" in txt.lower() and "iloc" not in txt:
                return False, "Los valores parecen correctos pero el índice no coincide. ¿Hace falta ordenar o usar reset_index()?"
            if ordered and not ignore_index:
                try:
                    pd.testing.assert_frame_equal(
                        u2.to_frame() if isinstance(u2, pd.Series) else u2,
                        e2.to_frame() if isinstance(e2, pd.Series) else e2,
                        check_like=True, **kw)
                except AssertionError:
                    pass
                else:
                    return False, "Mismos datos, pero en otro orden. Revisa cómo ordenas las filas."
            return False, "Los valores no coinciden con lo esperado. Revisa la lógica y vuelve a intentarlo."
        return True, ""
    if isinstance(u, (pd.DataFrame, pd.Series)):
        return False, f"Tu resultado es {_tipo(u)}, pero se esperaba un valor simple ({type(e).__name__}). ¿Te falta .iloc[0], .sum() o similar?"
    if _scalar_eq(u, e):
        return True, ""
    return False, f"Tu resultado es {u!r}, pero no es el valor esperado."


def pq_check(ns, exp_ns, opts):
    """Evalúa el ejercicio. ns = namespace del alumno, exp_ns = namespace de la solución."""
    custom = opts.get("custom")
    if custom:
        try:
            ok = bool(eval(custom, ns))
        except Exception as err:
            return False, f"Aún no funciona: {type(err).__name__}: {err}"
        return ok, "" if ok else opts.get("custom_msg", "Todavía no es correcto. ¡Sigue intentándolo!")
    if "resultado" not in ns:
        return False, "Guarda tu respuesta en la variable `resultado`."
    return pq_compare(ns["resultado"], exp_ns["resultado"],
                      ordered=opts.get("ordered", True),
                      ignore_index=opts.get("ignore_index", False))


_ERR_TIPS = [
    ("truth value of a Series is ambiguous", "Para combinar condiciones usa & (y), | (o) y pon cada condición entre paréntesis."),
    ("KeyError", "Revisa que el nombre de la columna esté bien escrito (mayúsculas, tildes, espacios)."),
    ("NameError", "Estás usando una variable que no existe. ¿Un error de tipeo?"),
    ("SyntaxError", "Hay un error de sintaxis: revisa paréntesis, comillas y que la línea `resultado = ...` esté completa."),
    ("AttributeError", "Ese método o atributo no existe para este objeto. Revisa el nombre."),
    ("TypeError", "Algún tipo de dato no encaja (¿texto mezclado con números?)."),
    ("IndentationError", "Revisa la sangría (espacios al inicio de la línea)."),
    ("MergeError", "Revisa las columnas que usas para unir (on / left_on / right_on)."),
]


def pq_error_tip(msg):
    for key, tip in _ERR_TIPS:
        if key in msg:
            return tip
    return ""


_SETUP = 'import pandas as pd\nimport numpy as np\n\nclientes = pd.DataFrame({\n    "cliente_id": range(1, 13),\n    "nombre": ["Ana", "Bruno", "Carla", "Diego", "Elena", "Facundo",\n               "Gabriela", "Hugo", "Inés", "Julián", "Karina", "Lucas"],\n    "ciudad": ["Madrid", "Buenos Aires", "CDMX", "Bogotá", "Madrid", "Lima",\n               "Buenos Aires", "CDMX", "Santiago", "Bogotá", "Madrid", "Lima"],\n    "pais": ["España", "Argentina", "México", "Colombia", "España", "Perú",\n             "Argentina", "México", "Chile", "Colombia", "España", "Perú"],\n    "edad": [34, 28, 45, 23, 51, 39, 30, 27, 42, 36, 25, 48],\n    "segmento": ["Premium", "Básico", "Premium", "Básico", "Premium", "Estándar",\n                 "Estándar", "Básico", "Premium", "Estándar", "Básico", "Estándar"],\n    "email": ["ana@mail.com", None, "carla@mail.com", "diego@mail.com", None, "facu@mail.com",\n              "gabi@mail.com", None, "ines@mail.com", "julian@mail.com", "kari@mail.com", "lucas@mail.com"],\n    "fecha_registro": pd.to_datetime([\n        "2023-01-15", "2023-02-03", "2023-02-20", "2023-03-11", "2023-04-02", "2023-05-19",\n        "2023-06-07", "2023-07-23", "2023-08-30", "2023-09-14", "2023-10-05", "2023-11-28"]),\n})\n\nproductos = pd.DataFrame({\n    "producto_id": range(1, 11),\n    "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",\n                 "Webcam", "Silla gamer", "Escritorio", "Tablet", "Micrófono"],\n    "categoria": ["Computación", "Accesorios", "Accesorios", "Computación", "Audio",\n                  "Accesorios", "Muebles", "Muebles", "Computación", "Audio"],\n    "precio": [1200.0, 25.5, 45.0, 300.0, 80.0, 60.0, 250.0, 180.0, 450.0, 95.0],\n    "stock": [5, 150, 80, 20, 60, 0, 12, 7, 15, 0],\n})\n\n_estados = ["entregado", "enviado", "entregado", "cancelado", "entregado", "pendiente"]\npedidos = pd.DataFrame({\n    "pedido_id": range(1001, 1031),\n    "cliente_id": [(i * 7) % 11 + 1 for i in range(29)] + [99],\n    "producto_id": [(i * 3) % 10 + 1 for i in range(30)],\n    "cantidad": [(i % 4) + 1 for i in range(30)],\n    "fecha": pd.date_range("2024-01-03", periods=30, freq="5D"),\n    "estado": [_estados[i % 6] for i in range(30)],\n})\n\nempleados = pd.DataFrame({\n    "emp_id": range(1, 11),\n    "nombre": ["Marta", "Pablo", "Sofía", "Tomás", "Valeria",\n               "Andrés", "Lucía", "Martín", "Paula", "Ramiro"],\n    "depto": ["Dirección", "Ventas", "Ventas", "Ventas", "IT",\n              "IT", "IT", "Marketing", "Marketing", "Ventas"],\n    "salario": [9000, 4200, 3900, 4000, 5200, 4800, 6100, 3700, 4100, 3500],\n    "jefe_id": pd.array([None, 1, 2, 2, 1, 5, 5, 1, 8, 2], dtype="Int64"),\n    "fecha_ingreso": pd.to_datetime([\n        "2015-03-01", "2018-06-15", "2019-01-10", "2020-09-01", "2017-11-20",\n        "2021-02-01", "2016-08-08", "2022-04-18", "2019-07-07", "2023-01-09"]),\n})\n\n_i = np.arange(90)\n_f = pd.date_range("2024-01-01", periods=90, freq="D")\n_v = 1000 + 8 * _i + 250 * (_f.dayofweek >= 5) + ((_i * 37) % 11) * 20\n_v = np.where(np.isin(_i, [20, 55, 77]), _v + 1500, _v)\nventas_diarias = pd.DataFrame({\n    "fecha": _f,\n    "ventas": _v.astype(float),\n    "visitas": (300 + (_i * 13) % 97 + 2 * _i).astype(int),\n})\n\nsucio = pd.DataFrame({\n    "id": [1, 2, 2, 3, 4, 5, 6, 7],\n    "nombre": ["  ana ", "BRUNO", "BRUNO", "carla", None, "Diego  ", "elena", "facundo"],\n    "edad": ["34", "28", "28", "n/d", "23", "51", None, "39"],\n    "ciudad": ["madrid", "Buenos Aires", "Buenos Aires", " santiago", "bogotá", "Madrid ", "madrid", "LIMA"],\n    "monto": ["$1,200.50", "$300", "$300", "$45.00", None, "$80.5", "$99", "$1,000"],\n})\ndel _i, _f, _v, _estados\n'
_SOL_B64 = 'eyJtMDdlMSI6IHsic29sdXRpb24iOiAicmVzdWx0YWRvID0gcGVkaWRvcy5tZXJnZShwcm9kdWN0b3MsIG9uPSdwcm9kdWN0b19pZCcpIiwgImNoZWNrIjogeyJvcmRlcmVkIjogZmFsc2UsICJpZ25vcmVfaW5kZXgiOiB0cnVlfSwgInhwIjogMjB9LCAibTA3ZTIiOiB7InNvbHV0aW9uIjogImRmID0gcGVkaWRvcy5tZXJnZShwcm9kdWN0b3MsIG9uPSdwcm9kdWN0b19pZCcpXG5kZiA9IGRmW2RmWydlc3RhZG8nXSAhPSAnY2FuY2VsYWRvJ11cbnJlc3VsdGFkbyA9IChkZlsnY2FudGlkYWQnXSAqIGRmWydwcmVjaW8nXSkuc3VtKCkiLCAiY2hlY2siOiB7fSwgInhwIjogMjB9LCAibTA3ZTMiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IGNsaWVudGVzW35jbGllbnRlc1snY2xpZW50ZV9pZCddLmlzaW4ocGVkaWRvc1snY2xpZW50ZV9pZCddKV0iLCAiY2hlY2siOiB7fSwgInhwIjogMzV9LCAibTA3ZTQiOiB7InNvbHV0aW9uIjogInJlc3VsdGFkbyA9IHBlZGlkb3NbfnBlZGlkb3NbJ2NsaWVudGVfaWQnXS5pc2luKGNsaWVudGVzWydjbGllbnRlX2lkJ10pXSIsICJjaGVjayI6IHsib3JkZXJlZCI6IGZhbHNlLCAiaWdub3JlX2luZGV4IjogdHJ1ZX0sICJ4cCI6IDM1fSwgIm0wN2U1IjogeyJzb2x1dGlvbiI6ICJkZiA9IChwZWRpZG9zXG4gICAgICAubWVyZ2UoY2xpZW50ZXMsIG9uPSdjbGllbnRlX2lkJylcbiAgICAgIC5tZXJnZShwcm9kdWN0b3MsIG9uPSdwcm9kdWN0b19pZCcpKVxuZGYgPSBkZltkZlsnZXN0YWRvJ10gIT0gJ2NhbmNlbGFkbyddXG5kZlsnaW1wb3J0ZSddID0gZGZbJ2NhbnRpZGFkJ10gKiBkZlsncHJlY2lvJ11cbnJlc3VsdGFkbyA9IGRmLmdyb3VwYnkoJ3BhaXMnKVsnaW1wb3J0ZSddLnN1bSgpLnNvcnRfdmFsdWVzKGFzY2VuZGluZz1GYWxzZSkiLCAiY2hlY2siOiB7fSwgInhwIjogMzV9LCAibTA3ZTYiOiB7InNvbHV0aW9uIjogIm0gPSBlbXBsZWFkb3MubWVyZ2UoZW1wbGVhZG9zLCBsZWZ0X29uPSdqZWZlX2lkJywgcmlnaHRfb249J2VtcF9pZCcsIGhvdz0nbGVmdCcsIHN1ZmZpeGVzPSgnJywgJ19qZWZlJykpXG5yZXN1bHRhZG8gPSBtW1snbm9tYnJlJywgJ25vbWJyZV9qZWZlJ11dLnJlbmFtZShjb2x1bW5zPXsnbm9tYnJlX2plZmUnOiAnamVmZSd9KSIsICJjaGVjayI6IHsib3JkZXJlZCI6IGZhbHNlLCAiaWdub3JlX2luZGV4IjogdHJ1ZX0sICJ4cCI6IDUwfSwgIm0wN2U3IjogeyJzb2x1dGlvbiI6ICJlc3BhbmEgPSBjbGllbnRlc1tjbGllbnRlc1sncGFpcyddID09ICdFc3Bhw7FhJ11cbnBlcnUgPSBjbGllbnRlc1tjbGllbnRlc1sncGFpcyddID09ICdQZXLDuiddXG5yZXN1bHRhZG8gPSBwZC5jb25jYXQoW2VzcGFuYSwgcGVydV0sIGlnbm9yZV9pbmRleD1UcnVlKSIsICJjaGVjayI6IHt9LCAieHAiOiAyMH19'
_TABLAS = ['clientes', 'productos', 'pedidos', 'empleados', 'ventas_diarias', 'sucio']

import base64 as _b64, json as _json, re as _re, warnings as _w
_w.simplefilter("ignore")
_SOL = _json.loads(_b64.b64decode(_SOL_B64).decode("utf-8"))

def reiniciar_datos():
    """Vuelve a cargar todas las tablas limpias (cada ejercicio empieza de cero)."""
    globals().pop("resultado", None)
    exec(_SETUP, globals())

def _codigo_celda():
    try:
        src = get_ipython().history_manager.input_hist_raw[-1]
    except Exception:
        return ""
    src = "\n".join(l for l in src.splitlines() if "comprobar(" not in l)
    return "\n".join(_re.sub(r"#.*$", "", l) for l in src.splitlines())

def comprobar(ej):
    info = _SOL[ej]
    exp_ns = {}
    exec(_SETUP, exp_ns)
    exec(info["solution"], exp_ns)
    g = globals()
    g["_pq_code"] = _codigo_celda()
    ok, msg = pq_check(g, exp_ns, info["check"])
    if ok:
        from IPython.display import HTML, display
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#0f9d6e;color:#fff;'
                     'font:600 15px system-ui">🎉 ¡Correcto! +' + str(info["xp"]) + ' XP · Sigue así 🐼</div>'))
    else:
        from IPython.display import HTML, display
        import html as _h
        display(HTML('<div style="padding:12px 16px;border-radius:12px;background:#b4235a;color:#fff;'
                     'font:500 15px system-ui">❌ Todavía no. ' + _h.escape(msg) + '</div>'))
    if "resultado" in g:
        return g["resultado"]

def ver_solucion(ej):
    print(_SOL[ej]["solution"])

reiniciar_datos()
print("✅ Datos cargados:", ", ".join(_TABLAS))


---
## 1. Inner join
`Intermedio` · **+20 XP**

`df1.merge(df2, on="clave")` une dos tablas por una columna común. Por defecto es un **inner join**: solo quedan las filas con coincidencia en ambas.

### 🎯 Tu misión
Une `pedidos` con `productos` por `producto_id`.

<details><summary>💡 Pista</summary>

`pedidos.merge(productos, on='producto_id')`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT * FROM pedidos p
JOIN productos pr ON p.producto_id = pr.producto_id;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("m07e1")

---
## 2. ¿Cuánto facturamos?
`Intermedio` · **+20 XP**

Tras unir, puedes calcular columnas con datos de ambas tablas y agregarlas. Recuerda filtrar lo que no cuenta como venta.

### 🎯 Tu misión
Facturación total (`cantidad * precio`) de los pedidos que **no** están `cancelado`. Guarda un número.

<details><summary>💡 Pista</summary>

`Filtra estado != 'cancelado' y luego (cantidad * precio).sum()`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT SUM(p.cantidad * pr.precio)
FROM pedidos p JOIN productos pr USING (producto_id)
WHERE p.estado <> 'cancelado';
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

df = pedidos.merge(productos, on='producto_id')
resultado = 

comprobar("m07e2")

---
## 3. Clientes sin pedidos
`Avanzado` · **+35 XP**

Un **anti join** busca filas de A sin pareja en B. En pandas lo más directo es `isin` negado:

```python
a[~a["id"].isin(b["id"])]
```
También puedes hacer un `merge(how="left", indicator=True)` y filtrar `_merge == "left_only"`.

### 🎯 Tu misión
Clientes que **nunca** hicieron un pedido.

<details><summary>💡 Pista</summary>

`~clientes['cliente_id'].isin(pedidos['cliente_id'])`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT c.* FROM clientes c
LEFT JOIN pedidos p ON c.cliente_id = p.cliente_id
WHERE p.pedido_id IS NULL;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

resultado = 

comprobar("m07e3")

---
## 4. Pedidos huérfanos
`Avanzado` · **+35 XP**

Problema real de calidad de datos: filas que apuntan a claves que no existen (integridad referencial rota). `indicator=True` añade la columna `_merge` con `both`, `left_only` o `right_only`.

### 🎯 Tu misión
Encuentra los pedidos cuyo `cliente_id` **no existe** en `clientes`. Devuelve solo las columnas originales de `pedidos`.

<details><summary>💡 Pista</summary>

`Filtra m['_merge'] == 'left_only' y quédate con pedidos.columns, o usa ~isin`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT p.* FROM pedidos p
LEFT JOIN clientes c ON p.cliente_id = c.cliente_id
WHERE c.cliente_id IS NULL;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

m = pedidos.merge(clientes, on='cliente_id', how='left', indicator=True)
resultado = 

comprobar("m07e4")

---
## 5. Ventas por país
`Avanzado` · **+35 XP**

Puedes encadenar varios `merge` como encadenas JOINs. Piensa en el grano: una fila por pedido → luego agregas.

### 🎯 Tu misión
Ingresos (`cantidad * precio`) por `pais` del cliente, **sin** pedidos cancelados, como Series ordenada de mayor a menor.

<details><summary>💡 Pista</summary>

`Filtra cancelados, crea importe, groupby('pais')['importe'].sum().sort_values(ascending=False)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT c.pais, SUM(p.cantidad * pr.precio) AS ingresos
FROM pedidos p
JOIN clientes c USING (cliente_id)
JOIN productos pr USING (producto_id)
WHERE p.estado <> 'cancelado'
GROUP BY c.pais ORDER BY ingresos DESC;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

df = (pedidos
      .merge(clientes, on='cliente_id')
      .merge(productos, on='producto_id'))


comprobar("m07e5")

---
## 6. ¿Quién es mi jefe?
`Pro` · **+50 XP**

Un **self join** une una tabla consigo misma. Cuando las claves tienen nombres distintos usa `left_on` / `right_on`, y `suffixes` para distinguir columnas repetidas.

### 🎯 Tu misión
DataFrame con columnas `nombre` (empleado) y `jefe` (nombre de su jefe) para **todos** los empleados; quien no tiene jefe queda con nulo.

<details><summary>💡 Pista</summary>

`Selecciona ['nombre', 'nombre_jefe'] y renombra a 'jefe'`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT e.nombre, j.nombre AS jefe
FROM empleados e
LEFT JOIN empleados j ON e.jefe_id = j.emp_id;
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

m = empleados.merge(empleados, left_on='jefe_id', right_on='emp_id', how='left', suffixes=('', '_jefe'))
resultado = 

comprobar("m07e6")

---
## 7. Apilar tablas
`Intermedio` · **+20 XP**

`pd.concat([df1, df2])` apila DataFrames uno debajo del otro (como `UNION ALL`). Con `ignore_index=True` se renumera el índice.

### 🎯 Tu misión
Apila los clientes de **España** y luego los de **Perú**, con el índice renumerado desde 0.

<details><summary>💡 Pista</summary>

`pd.concat([espana, peru], ignore_index=True)`
</details>

<details><summary>🗄️ Equivalente en SQL</summary>

```sql
SELECT * FROM clientes WHERE pais = 'España'
UNION ALL
SELECT * FROM clientes WHERE pais = 'Perú';
```
</details>

In [ ]:
reiniciar_datos()  # empieza con los datos limpios

espana = clientes[clientes['pais'] == 'España']
peru = clientes[clientes['pais'] == 'Perú']
resultado = 

comprobar("m07e7")

---
# 🏁 ¡Módulo completado!
Vuelve a [PandasQuest](https://rosalesluciano.github.io/pandas-quest/) para sumar tus XP y seguir con el siguiente módulo. 🚀